In [10]:
from langchain_community.document_loaders import DirectoryLoader
pdfs = DirectoryLoader("documentos", glob="*.pdf").load()
len(pdfs)

3

In [11]:
from transformers import AutoTokenizer 
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")


In [12]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=1250, chunk_overlap=150
)

pedacos = splitter.split_documents(pdfs)
len(pedacos)

37

In [14]:
pip install faiss-cpu --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 32.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [15]:

from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="bge-m3:567m")
vector_store = FAISS.from_documents(
    documents=pedacos,
    embedding=embeddings
)


In [22]:
retriever = vector_store.as_retriever()

from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda usando exclusivamente os conteúdos fornecdiso. \n\nContexto:\n{contexto}"),
        ("human", "{query}")
    ]
)

from langchain_ollama.llms import OllamaLLM

modelo = OllamaLLM(model="gemma3:4b")

from langchain_core.output_parsers import StrOutputParser
cadeia = prompt | modelo | StrOutputParser()



In [ ]:
pergunta = "como fazer um seguro viagem?"

cadeia.invoke(pergunta)

'Fazer um seguro viagem pode parecer complicado no início, mas com as informações certas, o processo se torna bem simples e tranquilo. Aqui está um guia completo sobre como fazer um seguro viagem:\n\n**1. Entenda suas Necessidades:**\n\n*   **Destino:** O país ou países que você vai visitar são cruciais, pois a cobertura do seguro varia conforme a região.\n*   **Duração da Viagem:** Determine o número de dias da sua viagem para calcular o prêmio do seguro.\n*   **Número de Viajantes:** O seguro é individual ou para um grupo?\n*   **Motivos da Viagem:** É uma viagem de lazer, negócios, estudo ou outra? Alguns seguros oferecem coberturas específicas para cada tipo de viagem.\n*   **Atividades:** Quais atividades você pretende fazer? Mergulho, esqui, rafting, esportes radicais, etc., podem exigir coberturas adicionais.\n*   **Idade:** A idade do(a) viajante influencia o preço do seguro.\n\n**2. Tipos de Cobertura:**\n\n*   **Assistência Médica e Hospitalar:** A cobertura mais básica, que 

In [23]:
trechos = retriever.invoke(pergunta)

contexto = "\n\n".join(trecho.page_content for trecho in trechos)

cadeia.invoke({"query": pergunta, "contexto": contexto})

'Com base no documento fornecido, aqui está um guia sobre como obter um seguro viagem, detalhando as opções e os passos:\n\n**Opções de Seguro Viagem:**\n\nO documento descreve dois tipos principais de seguro viagem oferecidos pelo Mastercard:\n\n*   **MasterAssist Plus:** Este é o seguro abrangente que oferece uma variedade de coberturas, incluindo:\n    *   Despesas Médicas e Hospitalares em Viagem ao Exterior (Acidente ou Doença Súbita)\n    *   Traslado Médico (Remoção Médica)\n    *   Traslado de Corpo (Repatriação Funerária)\n    *   Retorno de Menores/Idosos\n    *   Acompanhante em caso de hospitalização prolongada\n    *   Hospedagem\n*   **Coberturas Específicas:** O Mastercard também oferece proteção de compra (Compra Protegida) e garantia estendida original (Seguro Garantia Estendida Original), que podem ser acionadas em caso de danos à propriedade adquirida com o cartão.\n\n**Como Obter o Seguro Viagem (MasterAssist Plus):**\n\n1.  **Verificar a Elegibilidade:** O seguro M

# REWRITE-RETRIEVE-READ

In [61]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "contexto":  RunnablePassthrough() | retriever,
        "query": RunnablePassthrough()
    }
    | prompt | modelo | StrOutputParser()
)

In [62]:
rag_chain.invoke(pergunta)

KeyboardInterrupt: 

In [ ]:
query_model = OllamaLLM(model="gemma3:1b")

rewriter_prompt_template = """
Gere consulta de pesquisa para o banco de dados de vetores (Vector DB) a partir de uma pergunta do usuário,
permitindo uma resposta mais precisa por meio da busca semântica.
Basta retornar a consulta revisada do Vector DB, entre aspas.

Pergunta do usuário: {user_question}
Consulta revisada do Vector DB:
""" 

from langchain_core.prompts import PromptTemplate

rewriter_prompt_template = PromptTemplate.from_template(rewriter_prompt_template)

rewrite_chain = rewriter_prompt_template | query_model | StrOutputParser()

print(pergunta)
rewrite_chain.invoke(pergunta)   

como fazer um seguro viagem?


'```json\n{\n  "query": "seguro viagem",\n  "suggestions": [\n    "seguro viagem completo",\n    "seguro viagem para [país]",\n    "seguro viagem barato",\n    "comparar seguros viagem",\n    "como contratar seguro viagem",\n    "melhores seguros viagem",\n    "preço seguro viagem",\n    "benefícios seguro viagem",\n    "seguro viagem para [região]"\n  ]\n}\n```\n'

In [ ]:
rewriter_rag_chain = (
    {
        "contexto":  RunnablePassthrough() | rewrite_chain  | retriever,
        "query": RunnablePassthrough()
    }
    | prompt | modelo | StrOutputParser()
)

In [ ]:
rewriter_rag_chain.invoke(pergunta)

'Com base nos documentos fornecidos, aqui está um resumo de como obter um seguro viagem, com base nas informações contidas neles:\n\n**1. Tipos de Seguro Viagem:**\n\n*   **Mastercard Global Service (Seguro Viagem):**  Esses seguros são oferecidos em conjunto com cartões Mastercard. Eles cobrem diversas situações, como despesas médicas, traslado médico, perda de bens e outros imprevistos durante a viagem. Os documentos fornecem detalhes sobre seguros Gold e Platinum, indicando diferentes limites de cobertura (despesas médicas de até USD 25.000 ou USD 50.000, por exemplo) e coberturas adicionais como traslado médico e retorno de menores/idosos.\n*   **Serviços sem Desembolso de Dinheiro:** Com o seguro Mastercard, pode ser possível que a Mastercard negocie diretamente com hospitais e clínicas, pagando as despesas médicas sem que você precise pagar antecipadamente.\n\n**2. Coberturas e Limites de Cobertura (com base nos documentos):**\n\n*   **Despesas Médicas e Hospitalares:** Até USD 2

# MULTI-QUERY

In [ ]:
multi_query_prompt_template = """Você é um assistente de modelo de linguagem de IA. Sua tarefa é gerar cinco
versões diferentes da pergunta do usuário para recuperar documentos relevantes de um banco de dados vetorial.
Ao gerar múltiplas perspectivas sobre a pergunta do usuário, seu objetivo é ajudar
o usuário a superar algumas das limitações da busca por similaridade baseada em distância.
Forneça estas perguntas alternativas separadas por quebras de linha.
Pergunta original: {question}"""

multi_query_prompt = PromptTemplate.from_template(multi_query_prompt_template)

from langchain_core.output_parsers import CommaSeparatedListOutputParser

multi_query_chain = multi_query_prompt | modelo | CommaSeparatedListOutputParser()

print(pergunta)
multi_query_chain.invoke(pergunta)

como fazer um seguro viagem?


['Aqui estão cinco versões alternativas da pergunta original ("como fazer um seguro viagem?")',
 'projetadas para recuperar documentos relevantes de um banco de dados vetorial de diferentes ângulos:',
 '1.  Processo de compra de apólices de seguro viagem: quais são os passos e requisitos para adquirir uma cobertura de seguro viagem?',
 '2.  Requisitos para seguro viagem: quais são os fatores a considerar ao escolher um seguro viagem (idade',
 'destino',
 'atividades',
 'etc.) e como isso afeta o custo e a cobertura?',
 '3.  Comparação de seguros de viagem: quais são as diferentes seguradoras de viagem disponíveis e como compará-las em termos de preço',
 'cobertura',
 'exclusões e serviços adicionais?',
 '4.  Cobertura de seguro viagem: quais são os tipos de cobertura de seguro viagem que existem (viagem médica',
 'cancelamento de viagem',
 'perda de bagagem',
 'etc.) e como selecionar a cobertura mais adequada às minhas necessidades?',
 '5.  Documentos necessários para seguro viagem: q

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_retriever = MultiQueryRetriever(
    retriever=retriever, llm_chain=multi_query_chain
)

multi_rag_chain = (
    {
        "contexto": RunnablePassthrough() | multi_retriever,
        "query": RunnablePassthrough()
    }
    | prompt | modelo | StrOutputParser()
)


ImportError: cannot import name 'MultiQueryRetriever' from 'langchain_community.retrievers' (/Users/stell/Documents/ALURA/LangChain-Tecnicas-Avancas-de-RAG-main/env/lib/python3.14/site-packages/langchain_community/retrievers/__init__.py)

# HYDE

In [ ]:
hyde_prompt_template = """
Escreva um parágrafo que possa responder a pergunta apresentada. Não adicione informações.
Pergunta: {user_question}
Paragrafo:
""" 
hyde_prompt = PromptTemplate.from_template(hyde_prompt_template)
hyde_chain = hyde_prompt | modelo | StrOutputParser()

hyde_chain.invoke(pergunta)   

'Para fazer um seguro viagem, você deve pesquisar diferentes seguradoras, comparar coberturas, analisar o preço e escolher o plano que melhor atende às suas necessidades. Após a escolha, você precisará preencher um formulário de inscrição, fornecendo informações pessoais e de contato, e efetuar o pagamento da primeira mensalidade ou seguro.'